# Grounded Property Content Evaluation

This notebook reviews both the reproducible offline control and the selected recorded live Inspect run. Reading the recorded run requires no API credentials and makes no model calls.

In [1]:
from property_content.offline_report import build_offline_summary

summary = await build_offline_summary()
summary["aggregate"]

{'fixture_count': 4,
 'mandatory_pass_count': 4,
 'minimum_grounded_statement_rate': 1.0,
 'minimum_description_fact_support_rate': 1.0}

In [2]:
for sample in summary["samples"]:
    print(
        f"{sample['case_id']}: mandatory_criteria={sample['mandatory_criteria_passed']}, grounded={sample['grounded_statement_rate']:.0%}"
    )

description_rich: mandatory_criteria=True, grounded=100%
sparse: mandatory_criteria=True, grounded=100%
conflicting_capacity: mandatory_criteria=True, grounded=100%
adversarial_holdout: mandatory_criteria=True, grounded=100%


In [3]:
sample = summary["samples"][0]
sample["output"], sample["grounded_output"], sample["checks"]

({'hero_headline': 'Loft stay at Casa Llum in Barcelona',
  'property_highlights': ['A Loft located in Barcelona, Spain.',
   'Space for up to 4 guests across 2 bedrooms.',
   'The property provides 1.5 bathrooms.'],
  'about_this_place': 'Casa Llum is a Loft in Barcelona, Spain, with space for as many as 4 guests. Its layout includes 2 bedrooms and 1.5 bathrooms, giving guests the core details needed to plan their stay. The stated arrival and departure schedule is check-in at 3 PM and check-out at 11 AM, giving guests the supplied times for planning both ends of the stay.',
  'amenities_descriptions': [{'amenity_code': 'AirConditioning',
    'text': 'The property includes Air conditioning.'},
   {'amenity_code': 'DishWasher', 'text': 'The property includes Dishwasher.'},
   {'amenity_code': 'InternetBroadband',
    'text': 'The property includes High-speed internet.'}]},
 {'hero_headline': {'text': 'Loft stay at Casa Llum in Barcelona',
   'fact_ids': ['property.type', 'property.name'

## Live Inspect results

The selected live run used Anthropic Claude Sonnet 4.5 for generation and grading. All four properties pass every mandatory evaluation criterion with 100% statement grounding, and all four pass the advisory repetition check. Two of four pass all five editorial dimensions; the two failures remain visible below as quality findings. Launch `uv run inspect view start --log-dir logs/final` for the full prompts, model events, fact catalogs, generated fields, scores, and explanations.

In [4]:
from inspect_ai.log import read_eval_log

LIVE_LOG = "logs/final/2026-08-14T13-52-17-00-00_property-content-eval_Qw43aR4DDWr4pzMbh3fU8S.eval"
CURRENT_METRICS = (
    "description_fact_support_rate",
    "schema_valid",
    "structure_valid",
    "citation_valid",
    "numeric_consistency",
    "conflict_free",
    "non_repetitive",
    "grounded_statement_rate",
    "grounding_passed",
    "editorial_quality_passed",
)
live_log = read_eval_log(LIVE_LOG)
live_scores = []
for sample in live_log.samples or []:
    recorded = dict(sample.scores["pipeline_evaluation"].value)
    live_scores.append(
        {"case_id": sample.id, **{metric: recorded[metric] for metric in CURRENT_METRICS}}
    )
live_scores

[{'case_id': 'adversarial_holdout',
  'description_fact_support_rate': 1.0,
  'schema_valid': 1,
  'structure_valid': 1,
  'citation_valid': 1,
  'numeric_consistency': 1,
  'conflict_free': 1,
  'non_repetitive': 1,
  'grounded_statement_rate': 1.0,
  'grounding_passed': 1,
  'editorial_quality_passed': 1},
 {'case_id': 'conflicting_capacity',
  'description_fact_support_rate': 1.0,
  'schema_valid': 1,
  'structure_valid': 1,
  'citation_valid': 1,
  'numeric_consistency': 1,
  'conflict_free': 1,
  'non_repetitive': 1,
  'grounded_statement_rate': 1.0,
  'grounding_passed': 1,
  'editorial_quality_passed': 1},
 {'case_id': 'description_rich',
  'description_fact_support_rate': 1.0,
  'schema_valid': 1,
  'structure_valid': 1,
  'citation_valid': 1,
  'numeric_consistency': 1,
  'conflict_free': 1,
  'non_repetitive': 1,
  'grounded_statement_rate': 1.0,
  'grounding_passed': 1,
  'editorial_quality_passed': 0},
 {'case_id': 'sparse',
  'description_fact_support_rate': 1.0,
  'schema

In [5]:
import json

live_sample = (live_log.samples or [])[0]
live_score = live_sample.scores["pipeline_evaluation"]
live_explanation = json.loads(live_score.explanation)
{
    "case_id": live_sample.id,
    "fact_catalog": live_score.metadata["property_facts"]["all_facts"],
    "generated_content": live_score.metadata["generation_result"]["content"],
    "evaluation_metrics": dict(live_score.value),
    "editorial_quality": live_explanation["editorial_quality"],
}

{'case_id': 'adversarial_holdout',
 'fact_catalog': [{'id': 'amenity.kitchenanddining',
   'category': 'amenity',
   'value': 'KitchenAndDining',
   'text': 'Includes Kitchen and dining facilities.',
   'source': 'structured',
   'source_path': 'amenities[0]',
   'source_quote': None,
   'usage': 'marketing',
   'status': 'active',
   'priority': 'high',
   'suitable_sections': ['property_highlights',
    'about_this_place',
    'amenities_descriptions'],
   'exclusion_reason': None},
  {'id': 'amenity.superfastteleport',
   'category': 'amenity',
   'value': 'SuperFastTeleport',
   'text': 'Unknown amenity code: SuperFastTeleport',
   'source': 'structured',
   'source_path': 'amenities[1]',
   'source_quote': None,
   'usage': 'internal_only',
   'status': 'excluded',
   'priority': 'low',
   'suitable_sections': [],
   'exclusion_reason': 'unknown amenity code'},
  {'id': 'description.fact.1036527e94a7',
   'category': 'outdoor_space.garden',
   'value': 'fenced garden',
   'text': 